In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:

DATA_PATH = "talabat_enhanced_orders2.csv"  # ensure the file is in the same folder as this notebook
df = pd.read_csv(DATA_PATH)

df.head(10)

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count").T)

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
target_col = "Order_Status"
df[target_col].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Count")
plt.show()

In [ ]:
def run_experiment(top_k):
    df_exp = df.copy()

In [ ]:
    df_exp["Order_Time"] = pd.to_datetime(df_exp["Order_Time"], errors="coerce")

    df_exp["order_hour"] = df_exp["Order_Time"].dt.hour
    df_exp["order_dayofweek"] = df_exp["Order_Time"].dt.dayofweek
    df_exp["is_weekend"] = df_exp["order_dayofweek"].isin([5,6]).astype(int)

    df_exp["is_peak_hour"] = df_exp["order_hour"].isin(
        list(range(12,16)) + list(range(19,24))
    ).astype(int)

    df_exp["price_per_item"] = df_exp["Total_Price"] / df_exp["Quantity"]

    df_exp["price_tier"] = pd.cut(
        df_exp["Total_Price"],
        bins=[0, 100, 250, 500, np.inf],
        labels=["low","medium","high","very_high"]
    )


In [ ]:
    top_items = df_exp["Item_Name"].value_counts().head(top_k).index
    df_exp["Item_Name_reduced"] = np.where(
        df_exp["Item_Name"].isin(top_items),
        df_exp["Item_Name"],
        "Other"
    )

In [ ]:
    drop_cols = ["Order_ID","User_ID","Restaurant_ID","Driver_ID","Order_Time","Item_Name"]
    drop_cols = [c for c in drop_cols if c in df_exp.columns]

    X = df_exp.drop(columns=drop_cols + ["Order_Status"])
    y = df_exp["Order_Status"]

In [ ]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

In [ ]:
    categorical_cols = X_train.select_dtypes(include=["object","category"]).columns.tolist()
    numeric_cols = X_train.select_dtypes(include=[np.number,"bool"]).columns.tolist()

In [ ]:
    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ])

    model = Pipeline([
        ("preprocess", preprocess),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=42))
    ])

In [ ]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

In [ ]:
    ohe = model.named_steps["preprocess"].named_transformers_["cat"]
    cat_names = ohe.get_feature_names_out(categorical_cols)
    all_features = np.concatenate([cat_names, numeric_cols])

    importances = model.named_steps["rf"].feature_importances_

    fi = pd.DataFrame({
        "feature": all_features,
        "importance": importances
    }).sort_values(by="importance", ascending=False)

In [ ]:
return acc, fi.head(5)

In [ ]:
for k in [3, 5, 6]:
    acc, fi = run_experiment(k)

    print(f"\n===== top_k = {k} =====")
    print("Accuracy:", round(acc, 4))
    print("\nTop Features:")
    print(fi)

In [ ]:
comparison = pd.DataFrame({
    "top_k": list(results.keys()),
    "Accuracy": list(results.values())
})

print("\nFinal Comparison:")
display(comparison)
